# Framework final controlado — CNN/RAG/SLM para diagnóstico de motores por termografia

Notebook único para executar **Mistral-7B**, **Qwen2.5-3B**, **TinyLlama-1.1B** e **Zephyr-7B** sob o mesmo protocolo final. Troque somente `SELECTED_MODEL` entre as quatro execuções.


## 1. Instalação

Execute esta célula, reinicie o runtime e continue a partir da seção 2.

In [1]:
# ============================================================
# CÉLULA 1 — INSTALAÇÃO FIXADA
# ============================================================
!pip install -q \
  pandas==2.2.2 \
  transformers==4.45.2 \
  accelerate==0.34.2 \
  bitsandbytes==0.49.2 \
  sentence-transformers==3.1.1 \
  evaluate==0.4.3 \
  rouge-score==0.1.2 \
  nltk==3.9.1 \
  scikit-learn==1.5.2 \
  faiss-cpu \
  sentencepiece \
  huggingface_hub \
  tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [ ]:
# ============================================================
# CÉLULA 1.1 — REINICIAR RUNTIME
# ============================================================
# Execute uma vez depois da instalação.
import os
os.kill(os.getpid(), 9)

## 2. Imports e verificação do ambiente

In [1]:
# ============================================================
# HUGGING FACE — CONFIGURAÇÃO DE DOWNLOAD
# Execute antes de importar huggingface_hub, transformers
# ou sentence_transformers.
# ============================================================

import os

# Desativa o backend Xet, que apresentou URLs assinadas inválidas.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Aumenta os tempos de espera para arquivos grandes.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

# Mantém a saída limpa.
os.environ["HF_HUB_VERBOSITY"] = "warning"

print("Configuração de download do Hugging Face aplicada.")
print("HF_HUB_DISABLE_XET =", os.environ["HF_HUB_DISABLE_XET"])


Configuração de download do Hugging Face aplicada.
HF_HUB_DISABLE_XET = 1


In [2]:
# ============================================================
# LOGIN SEGURO NO HUGGING FACE
# O Secret do Colab deve se chamar exatamente HF_token.
# ============================================================

from google.colab import userdata
from huggingface_hub import login, whoami
import os

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_token não encontrado nos Secrets do Colab. "
        "Habilite o acesso deste notebook ao Secret."
    )

# Nome padrão reconhecido internamente pelas bibliotecas do Hugging Face.
os.environ["HF_TOKEN"] = hf_token

login(
    token=hf_token,
    add_to_git_credential=False
)

hf_account = whoami(token=hf_token)

print("Login no Hugging Face realizado com sucesso.")
print("Usuário autenticado:", hf_account.get("name", "não identificado"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login no Hugging Face realizado com sucesso.
Usuário autenticado: LuizDahmer


In [4]:
from pathlib import Path
import os
import json
import time
import gc
import textwrap
import hashlib
import platform
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from sentence_transformers import SentenceTransformer
import faiss

print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



torch: 2.11.0+cu128
CUDA disponível: True
GPU: Tesla T4


## 3. Configuração central do experimento

Troque apenas `SELECTED_MODEL`. Opções: `mistral`, `qwen`, `tinyllama`, `zephyr`.

In [5]:
# ============================================================
# CONFIGURAÇÃO FINAL PADRONIZADA — V3 CNN + CONTEXTO LIMITADO
# Troque SOMENTE SELECTED_MODEL antes de cada execução.
# ============================================================
SELECTED_MODEL = "mistral"  # "mistral", "qwen", "tinyllama", "zephyr"

GLOBAL_CONFIG = {
    "experiment_version": "final_controlled_v3_cnn_limited_context",
    "embedding_model": "sentence-transformers/all-mpnet-base-v2",
    "rag_json_path": "/content/RAG_motores_eletricos.json",
    "cnn_predictions_path": "/content/holdout_test_predictions.csv",
    "rag_top_k": 3,

    # Limite comum aos quatro modelos.
    # O prompt é construído abaixo deste teto; a geração é abortada se ultrapassá-lo.
    "max_input_tokens": 1280,
    "prompt_safety_margin": 32,
    "minimum_context_tokens": 240,
    "results_dir": "/content/results_slm_rag_final",

    # Parâmetros IGUAIS para os quatro modelos
    "max_new_tokens": 700,
    "do_sample": True,
    "temperature": 0.2,
    "top_p": 0.9,
    "top_k": None,
    "repetition_penalty": 1.1,

    # Reprodutibilidade
    "base_seed": 42,

    # Controle de qualidade
    "minimum_report_chars": 120,
    "fail_on_truncation": True,
}

MODEL_PROFILES = {
    "mistral": {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
        "model_label": "Mistral-7B-Instruct-v0.2",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "mistral",
        "notes": "Mistral validado em 4-bit NF4; double_quant=False."
    },
    "qwen": {
        "model_name": "Qwen/Qwen2.5-3B-Instruct",
        "model_label": "Qwen2.5-3B-Instruct-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": True,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "qwen",
        "notes": "Qwen validado em FP16."
    },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "model_label": "TinyLlama-1.1B-Chat-v1.0-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": "left",
        "invalid_rules": "tinyllama",
        "notes": "TinyLlama validado em FP16."
    },
    "zephyr": {
        "model_name": "HuggingFaceH4/zephyr-7b-beta",
        "model_label": "Zephyr-7B-Beta-4bit",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "generic",
        "notes": "Zephyr em 4-bit NF4; double_quant=True."
    },
}

if SELECTED_MODEL not in MODEL_PROFILES:
    raise ValueError(f"Modelo inválido: {SELECTED_MODEL}")

EXPERIMENT_CONFIG = {**GLOBAL_CONFIG, **MODEL_PROFILES[SELECTED_MODEL]}

results_dir = Path(EXPERIMENT_CONFIG["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)

stem = f"slm_rag_results_{SELECTED_MODEL}_final"
EXPERIMENT_CONFIG["output_csv"] = str(results_dir / f"{stem}.csv")
EXPERIMENT_CONFIG["output_json"] = str(results_dir / f"{stem}.json")
EXPERIMENT_CONFIG["manifest_json"] = str(results_dir / f"manifest_{SELECTED_MODEL}_final.json")

print(json.dumps(EXPERIMENT_CONFIG, indent=2, ensure_ascii=False))


{
  "experiment_version": "final_controlled_v3_cnn_limited_context",
  "embedding_model": "sentence-transformers/all-mpnet-base-v2",
  "rag_json_path": "/content/RAG_motores_eletricos.json",
  "cnn_predictions_path": "/content/holdout_test_predictions.csv",
  "rag_top_k": 3,
  "max_input_tokens": 1280,
  "prompt_safety_margin": 32,
  "minimum_context_tokens": 240,
  "results_dir": "/content/results_slm_rag_final",
  "max_new_tokens": 700,
  "do_sample": true,
  "temperature": 0.2,
  "top_p": 0.9,
  "top_k": null,
  "repetition_penalty": 1.1,
  "base_seed": 42,
  "minimum_report_chars": 120,
  "fail_on_truncation": true,
  "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "model_label": "Mistral-7B-Instruct-v0.2",
  "loading_mode": "4bit_nf4",
  "load_in_4bit": true,
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_use_double_quant": false,
  "torch_dtype": "float16",
  "trust_remote_code": false,
  "use_fast_tokenizer": true,
  "padding_side": null

## 4. Upload/carregamento do JSON do RAG

In [6]:
from google.colab import files

rag_path = Path(EXPERIMENT_CONFIG["rag_json_path"])

if not rag_path.exists():
    print("Envie o arquivo RAG_motores_eletricos.json")
    uploaded = files.upload()
    for name in uploaded.keys():
        if name.endswith(".json"):
            Path(name).rename(rag_path)
            print(f"Arquivo salvo em: {rag_path}")
else:
    print(f"Arquivo encontrado: {rag_path}")

Envie o arquivo RAG_motores_eletricos.json


Saving RAG_motores_eletricos.json to RAG_motores_eletricos.json
Arquivo salvo em: /content/RAG_motores_eletricos.json


In [7]:
def load_rag_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

rag_data = load_rag_json(EXPERIMENT_CONFIG["rag_json_path"])
failures = rag_data["document"]["failures"]

print(f"Total de classes/falhas no RAG: {len(failures)}")
print([item.get("type") for item in failures])

Total de classes/falhas no RAG: 6
['Overheating', 'Ventilation Defect', 'Phase Loss', 'Blocked Rotor', 'Bearing Failure', 'Normal Operation']


## 5. Criação dos documentos do RAG

In [8]:
# ============================================================
# DOCUMENTOS RAG — 6 ENTRADAS ORIGINAIS (UMA POR CLASSE)
# A base não é dividida em 18 chunks. O limite é aplicado somente
# ao contexto efetivamente enviado ao modelo.
# ============================================================
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                parts.append("; ".join(f"{k}: {v}" for k, v in item.items()))
            else:
                parts.append(str(item))
        return "; ".join(parts)
    if isinstance(value, dict):
        return " | ".join(
            f"{key}: {as_text(item)}"
            for key, item in value.items()
        )
    return str(value)


documents = []

for fault in failures:
    fault_type = fault.get("type", "")

    parts = [
        f"Fault Type: {fault_type}",
        f"Aliases: {as_text(fault.get('aliases', []))}",
        f"Description: {as_text(fault.get('description'))}",
        f"Physical mechanism: {as_text(fault.get('physical_mechanism'))}",
        f"Root causes: {as_text(fault.get('root_causes'))}",
        f"Observable evidence: {as_text(fault.get('observable_evidence'))}",
        f"System effects: {as_text(fault.get('system_effects'))}",
        f"Maintenance actions: {as_text(fault.get('maintenance_actions'))}",
        f"Diagnostic checks: {as_text(fault.get('diagnostic_checks'))}",
        f"Risk level: {as_text(fault.get('risk_level'))}",
        f"Standards and guidelines: {as_text(fault.get('standards_and_guidelines'))}",
        f"RAG retrieval notes: {as_text(fault.get('rag_retrieval_notes'))}",
    ]

    text = "\n".join(p for p in parts if not p.endswith(": "))

    documents.append({
        "fault_type": fault_type,
        "section": "full_entry",
        "risk_level": fault.get("risk_level", ""),
        "text": text,
    })

print("Total de documentos para recuperação:", len(documents))
assert len(documents) == len(failures), "Esperava-se uma entrada RAG por falha."
print(documents[0]["text"][:1800])


Total de documentos para recuperação: 6
Fault Type: Overheating
Aliases: thermal overload; excessive temperature rise; abnormal heating; temperature rise; hot motor
Description: Overheating represents an abnormal thermal condition in which the motor operates above its expected thermal behavior for the applied load, ambient condition, duty cycle, and cooling configuration. In induction motors, this condition is critical because temperature rise is directly associated with accelerated insulation aging, winding degradation, reduction of dielectric strength, and increased probability of premature failure.
Physical mechanism: The fault is usually produced by an imbalance between heat generation and heat dissipation. Heat generation may increase due to overload, voltage imbalance, repeated starting, high current, winding defects, poor power quality, or mechanical friction. Heat dissipation may decrease due to obstructed ventilation, dirty cooling channels, damaged fans, high ambient temperat

## 6. Embeddings + FAISS

In [9]:
  # ============================================================
# EMBEDDING RAG — DOWNLOAD MÍNIMO E CARREGAMENTO LOCAL
# ============================================================

import time
from pathlib import Path

import faiss
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer


EMBEDDING_REPO = EXPERIMENT_CONFIG["embedding_model"]
EMBEDDING_LOCAL_DIR = Path("/content/models/all-mpnet-base-v2")

EMBEDDING_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Preparando embedding local:")
print(EMBEDDING_REPO)


# ------------------------------------------------------------
# SOMENTE OS ARQUIVOS NECESSÁRIOS PARA O SENTENCE-TRANSFORMERS
# ------------------------------------------------------------
required_files = [
    "modules.json",
    "config.json",
    "config_sentence_transformers.json",
    "sentence_bert_config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "tokenizer.json",
    "special_tokens_map.json",
    "vocab.txt",
    "1_Pooling/config.json",
]

# Alguns repositórios possuem este arquivo; caso não exista,
# ele não é indispensável.
optional_files = [
    "added_tokens.json",
]


def download_required_file(
    filename: str,
    maximum_attempts: int = 5
) -> str:
    """
    Baixa um arquivo específico e tenta novamente em caso de
    falha temporária do CDN do Hugging Face.
    """

    last_error = None

    for attempt in range(1, maximum_attempts + 1):

        try:
            print(
                f"Baixando {filename} "
                f"(tentativa {attempt}/{maximum_attempts})..."
            )

            downloaded_path = hf_hub_download(
                repo_id=EMBEDDING_REPO,
                filename=filename,
                local_dir=str(EMBEDDING_LOCAL_DIR),
                token=hf_token,
            )

            print("Concluído:", filename)

            return downloaded_path

        except Exception as error:
            last_error = error

            print(
                f"Falha ao baixar {filename}: "
                f"{type(error).__name__}"
            )

            if attempt < maximum_attempts:
                waiting_seconds = 5 * attempt

                print(
                    f"Nova tentativa em "
                    f"{waiting_seconds} segundos..."
                )

                time.sleep(waiting_seconds)

    raise RuntimeError(
        f"Não foi possível baixar o arquivo obrigatório "
        f"{filename} após {maximum_attempts} tentativas."
    ) from last_error


# ------------------------------------------------------------
# DOWNLOAD DOS ARQUIVOS OBRIGATÓRIOS
# ------------------------------------------------------------
for filename in required_files:
    download_required_file(filename)


# ------------------------------------------------------------
# DOWNLOAD DOS ARQUIVOS OPCIONAIS
# ------------------------------------------------------------
for filename in optional_files:

    try:
        hf_hub_download(
            repo_id=EMBEDDING_REPO,
            filename=filename,
            local_dir=str(EMBEDDING_LOCAL_DIR),
            token=hf_token,
        )

        print("Arquivo opcional concluído:", filename)

    except Exception:
        print(
            "Arquivo opcional não encontrado ou indisponível:",
            filename
        )


# ------------------------------------------------------------
# VALIDAÇÃO DO PESO PRINCIPAL
# ------------------------------------------------------------
weight_path = (
    EMBEDDING_LOCAL_DIR
    / "model.safetensors"
)

if not weight_path.exists():
    raise FileNotFoundError(
        f"Peso principal não encontrado: {weight_path}"
    )

weight_size_mb = (
    weight_path.stat().st_size
    / 1024**2
)

print(
    "Tamanho do model.safetensors:",
    round(weight_size_mb, 2),
    "MB"
)

if weight_size_mb < 400:
    raise RuntimeError(
        "O arquivo model.safetensors parece incompleto. "
        f"Tamanho encontrado: {weight_size_mb:.2f} MB."
    )


# ------------------------------------------------------------
# CARREGAMENTO EXCLUSIVAMENTE LOCAL
# ------------------------------------------------------------
embedding_model = SentenceTransformer(
    str(EMBEDDING_LOCAL_DIR),
    local_files_only=True,
)

print("Embedding carregado exclusivamente da pasta local.")


# ------------------------------------------------------------
# EMBEDDINGS E ÍNDICE FAISS
# ------------------------------------------------------------
texts = [
    doc["text"]
    for doc in documents
]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

index.add(embeddings)

print("Índice FAISS criado com sucesso.")
print("Documentos indexados:", index.ntotal)
print("Dimensão dos embeddings:", embeddings.shape[1])


# EMBEDDING_REPO = EXPERIMENT_CONFIG["embedding_model"]
# EMBEDDING_LOCAL_DIR = "/content/models/all-mpnet-base-v2"

# print("Preparando snapshot local do embedding:", EMBEDDING_REPO)

# embedding_snapshot_path = snapshot_download(
#     repo_id=EMBEDDING_REPO,
#     local_dir=EMBEDDING_LOCAL_DIR,
#     # local_dir_use_symlinks=False,  # Parâmetro deprecated, será ignorado
#     token=hf_token,
# )

# print("Snapshot do embedding disponível em:")
# print(embedding_snapshot_path)

# # A partir daqui, o SentenceTransformer trabalha apenas com arquivos locais.
# embedding_model = SentenceTransformer(
#     embedding_snapshot_path,
#     local_files_only=True,
# )

# texts = [doc["text"] for doc in documents]

# embeddings = embedding_model.encode(
#     texts,
#     convert_to_numpy=True,
#     normalize_embeddings=True,
#     show_progress_bar=True,
# ).astype("float32")

# index = faiss.IndexFlatIP(embeddings.shape[1])
# index.add(embeddings)

# print("Embedding carregado localmente:", EMBEDDING_REPO)
# print("Índice FAISS criado:", index.ntotal)


Preparando embedding local:
sentence-transformers/all-mpnet-base-v2
Baixando modules.json (tentativa 1/5)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

Concluído: modules.json
Baixando config.json (tentativa 1/5)...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

Concluído: config.json
Baixando config_sentence_transformers.json (tentativa 1/5)...


...

Concluído: vocab.txt
Baixando 1_Pooling/config.json (tentativa 1/5)...


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Concluído: 1_Pooling/config.json
Arquivo opcional não encontrado ou indisponível: added_tokens.json
Tamanho do model.safetensors: 417.68 MB
Embedding carregado exclusivamente da pasta local.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Índice FAISS criado com sucesso.
Documentos indexados: 6
Dimensão dos embeddings: 768


## 7. Recuperação RAG

In [10]:

CLASS_ALIASES = {
    "lack of phase": "Phase Loss",
    "phase loss": "Phase Loss",
    "normal": "Normal Operation",
    "normal operation": "Normal Operation",
    "bearing failure": "Bearing Failure",
    "blocked rotor": "Blocked Rotor",
    "overheating": "Overheating",
    "ventilation defect": "Ventilation Defect",
}

def canonicalize_class(name: str) -> str:
    key = str(name).strip().lower()
    return CLASS_ALIASES.get(key, str(name).strip())


def build_query(predicted_class: str, confidence: float, observed_evidence: str = "") -> str:
    canonical_class = canonicalize_class(predicted_class)
    return (
        f"Predicted fault: {canonical_class}. "
        f"CNN confidence: {confidence:.3f}. "
        f"Observed evidence: {observed_evidence}"
    )


def retrieve_context(
    predicted_class: str,
    confidence: float,
    observed_evidence: str = "",
    top_k: int = 3
) -> List[Dict[str, Any]]:

    query = build_query(predicted_class, confidence, observed_evidence)
    query_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_emb, top_k)

    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        doc = dict(documents[int(idx)])
        doc["score"] = float(score)
        retrieved.append(doc)

    return retrieved


## 8. Prompt técnico padronizado

In [11]:
def token_count(text: str, add_special_tokens: bool = False) -> int:
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=add_special_tokens
        )
    )


def trim_to_token_budget(text: str, token_budget: int) -> str:
    """Recorta pelo tokenizer do modelo, sem exceder o orçamento."""
    if token_budget <= 0:
        return ""

    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False
    )

    if len(token_ids) <= token_budget:
        return str(text).strip()

    return tokenizer.decode(
        token_ids[:token_budget],
        skip_special_tokens=True
    ).strip()


def format_context_block(
    position: int,
    doc: Dict[str, Any],
    body_text: str
) -> str:
    return (
        f"[Context {position}]\n"
        f"Fault type: {doc.get('fault_type', 'NA')}\n"
        f"Section: {doc.get('section', 'NA')}\n"
        f"Retrieval score: {doc.get('score', 0.0):.4f}\n"
        f"{body_text}"
    )


# def prompt_from_context(
#     canonical_class: str,
#     confidence: float,
#     evidence_text: str,
#     context_text: str
# ) -> str:
#     prompt = f"""
# You are an industrial maintenance specialist in induction motor fault diagnosis using infrared thermography.

# CNN output:
# - Predicted fault: {canonical_class}
# - CNN confidence: {confidence:.3f}

# Observed thermal/operational evidence:
# {evidence_text}

# Retrieved technical context from the RAG knowledge base:
# {context_text}

# Write one complete and concise technical diagnostic report in English.

# Required sections:
# 1. Technical diagnosis
# 2. Supporting evidence
# 3. Probable causes
# 4. Recommended maintenance actions
# 5. Severity and risk assessment

# Mandatory rules:
# - Use only information supported by the CNN output and retrieved context.
# - Clearly distinguish direct CNN outputs from probable or inferred causes.
# - Do not claim that the CNN observed textual thermal evidence when none was produced.
# - Do not invent measurements, standards, citations, or inspection results.
# - Prefer synthesis over copying long passages from the context.
# - Keep the complete report between approximately 250 and 450 words.
# - Finish all five sections and end with a complete sentence.
# """
#     return textwrap.dedent(prompt).strip()

def prompt_from_context(
    canonical_class: str,
    confidence: float,
    evidence_text: str,
    context_text: str
) -> str:

    prompt = f"""
You are an industrial maintenance specialist in induction motor fault diagnosis using infrared thermography.

CNN prediction:
- Predicted fault: {canonical_class}
- Confidence: {confidence:.3f}

Observed evidence:
{evidence_text}

Retrieved technical knowledge:
{context_text}

Generate ONE concise technical diagnostic report in English.

The report MUST contain EXACTLY the following five sections:

1. Technical diagnosis
2. Supporting evidence
3. Probable causes
4. Recommended maintenance actions
5. Severity and risk assessment

Mandatory rules:

- Use ONLY information supported by the CNN prediction and the retrieved context.
- Clearly distinguish confirmed information from probable causes.
- Never invent measurements, temperatures, standards, inspections or evidence.
- Do not state that the CNN observed textual thermal patterns unless they are explicitly provided.
- Summarize instead of copying the retrieved context.
- Write exactly five sections using ONLY the headings above.
- Use no more than THREE short sentences per section.
- Keep the entire report between approximately 180 and 300 words.
- Do NOT write introductions, conclusions, summaries, notes, bullet lists or additional sections.
- Do NOT repeat information across sections.
- Finish immediately after completing Section 5.
- End with one complete sentence.

Produce only the report.
"""

    return textwrap.dedent(prompt).strip()


def build_prompt(
    predicted_class: str,
    confidence: float,
    retrieved_docs: List[Dict[str, Any]],
    observed_evidence: str = ""
):
    """
    Mantém os 6 documentos originais no FAISS e top-k=3.
    Limita somente o texto dos documentos recuperados, usando o tokenizer
    do modelo selecionado e verificando o tamanho APÓS o chat template.

    Retorna:
      prompt_final,
      contexto_exato_fornecido,
      metadados_do_orçamento.
    """
    canonical_class = canonicalize_class(predicted_class)

    evidence_text = (
        str(observed_evidence).strip()
        if str(observed_evidence).strip()
        else "No textual thermal evidence was produced by the CNN; only the predicted class and confidence are available."
    )

    empty_prompt = prompt_from_context(
        canonical_class,
        confidence,
        evidence_text,
        context_text=""
    )
    empty_chat = apply_model_chat_template(empty_prompt)
    fixed_tokens = token_count(empty_chat, add_special_tokens=False)

    max_input_tokens = int(EXPERIMENT_CONFIG["max_input_tokens"])
    safety_margin = int(EXPERIMENT_CONFIG.get("prompt_safety_margin", 32))
    available_context_tokens = max_input_tokens - fixed_tokens - safety_margin

    minimum_context_tokens = int(
        EXPERIMENT_CONFIG.get("minimum_context_tokens", 240)
    )
    if available_context_tokens < minimum_context_tokens:
        raise RuntimeError(
            "Orçamento insuficiente para contexto: "
            f"{available_context_tokens} tokens disponíveis."
        )

    # Distribuição igual entre os top-k para não favorecer a posição do documento.
    per_document_budget = max(
        1,
        available_context_tokens // max(1, len(retrieved_docs))
    )

    blocks = []
    context_metadata = []

    for i, doc in enumerate(retrieved_docs, start=1):
        header = format_context_block(i, doc, body_text="")
        header_tokens = token_count(header, add_special_tokens=False)
        body_budget = max(1, per_document_budget - header_tokens)

        original_text = str(doc.get("text", ""))
        compact_text = trim_to_token_budget(original_text, body_budget)
        block = format_context_block(i, doc, compact_text)
        blocks.append(block)

        context_metadata.append({
            "rank": i,
            "fault_type": doc.get("fault_type", "NA"),
            "section": doc.get("section", "NA"),
            "retrieval_score": float(doc.get("score", 0.0)),
            "original_document_tokens": token_count(
                original_text,
                add_special_tokens=False
            ),
            "provided_document_tokens": token_count(
                compact_text,
                add_special_tokens=False
            ),
            "document_was_trimmed": compact_text.strip() != original_text.strip(),
        })

    context_text = "\n\n".join(blocks)
    prompt = prompt_from_context(
        canonical_class,
        confidence,
        evidence_text,
        context_text
    )

    # Ajuste final determinístico, considerando exatamente o chat template.
    chat_text = apply_model_chat_template(prompt)
    final_tokens = token_count(chat_text, add_special_tokens=False)
    target = max_input_tokens - safety_margin

    while final_tokens > target:
        # Retira um pequeno bloco de tokens do documento atualmente mais longo.
        longest_idx = max(
            range(len(context_metadata)),
            key=lambda idx: context_metadata[idx]["provided_document_tokens"]
        )
        current = context_metadata[longest_idx]["provided_document_tokens"]
        if current <= 24:
            raise RuntimeError(
                "Não foi possível reduzir o prompt ao limite sem remover "
                "praticamente todo o contexto."
            )

        new_budget = max(24, current - 16)
        doc = retrieved_docs[longest_idx]
        compact_text = trim_to_token_budget(
            str(doc.get("text", "")),
            new_budget
        )
        blocks[longest_idx] = format_context_block(
            longest_idx + 1,
            doc,
            compact_text
        )
        context_metadata[longest_idx]["provided_document_tokens"] = (
            token_count(compact_text, add_special_tokens=False)
        )
        context_metadata[longest_idx]["document_was_trimmed"] = True

        context_text = "\n\n".join(blocks)
        prompt = prompt_from_context(
            canonical_class,
            confidence,
            evidence_text,
            context_text
        )
        chat_text = apply_model_chat_template(prompt)
        final_tokens = token_count(chat_text, add_special_tokens=False)

    budget_metadata = {
        "max_input_tokens": max_input_tokens,
        "safety_margin": safety_margin,
        "fixed_prompt_tokens": fixed_tokens,
        "available_context_tokens": available_context_tokens,
        "final_chat_input_tokens": final_tokens,
        "documents": context_metadata,
    }

    return prompt, context_text, budget_metadata


In [12]:
# ============================================================
# UPLOAD DO CSV DE SAÍDA DA CNN (HOLD-OUT)
# ============================================================

from google.colab import files
import pandas as pd
from pathlib import Path

print("Selecione o arquivo holdout_test_predictions.csv")

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("Nenhum arquivo foi enviado.")

csv_name = list(uploaded.keys())[0]

# Caminho que será utilizado em todo o notebook
CNN_PREDICTIONS_PATH = Path("/content/holdout_test_predictions.csv")

# Renomeia automaticamente para manter o restante do código igual
Path(csv_name).rename(CNN_PREDICTIONS_PATH)

print(f"Arquivo salvo em: {CNN_PREDICTIONS_PATH}")

# Carrega para conferência
holdout_df = pd.read_csv(CNN_PREDICTIONS_PATH)

print("\nArquivo carregado com sucesso!")
print(f"Número de amostras: {len(holdout_df)}")
print(f"Número de colunas: {len(holdout_df.columns)}")

display(holdout_df.head())

Selecione o arquivo holdout_test_predictions.csv


Saving holdout_test_predictions.csv to holdout_test_predictions.csv
Arquivo salvo em: /content/holdout_test_predictions.csv

Arquivo carregado com sucesso!
Número de amostras: 500
Número de colunas: 9


,true_class,predicted_class,predicted_confidence,probability_Bearing_Failure,probability_Blocked_Rotor,probability_Lack_of_Phase,probability_Normal,probability_Overheating,probability_Ventilation_Defect
0,Ventilation_Defect,Ventilation_Defect,0.674806,0.080105,0.097212,0.061419,0.051824,0.034634,0.674806
1,Overheating,Overheating,0.483740,0.077910,0.095429,0.160605,0.085398,0.483740,0.096918
2,Blocked_Rotor,Blocked_Rotor,0.789369,0.043255,0.789369,0.015247,0.015795,0.112929,0.023405
3,Blocked_Rotor,Blocked_Rotor,0.489936,0.116453,0.489936,0.116758,0.088988,0.094018,0.093848
4,Blocked_Rotor,Blocked_Rotor,0.761209,0.035650,0.761209,0.057701,0.024385,0.058194,0.062861


## 9. Casos de teste padronizados

In [13]:
# ============================================================
# CASOS REAIS DA CNN — SELEÇÃO AUTOMÁTICA NO HOLD-OUT
# Uma amostra corretamente classificada por classe, cuja confiança
# seja a mais próxima da mediana da própria classe.
# ============================================================
CNN_CSV_PATH = EXPERIMENT_CONFIG["cnn_predictions_path"]

if not Path(CNN_CSV_PATH).exists():
    raise FileNotFoundError(
        f"Arquivo da CNN não encontrado: {CNN_CSV_PATH}. "
        "Envie holdout_test_predictions.csv para o Colab."
    )

holdout_df = pd.read_csv(CNN_CSV_PATH)

COLUMN_CANDIDATES = {
    "true_class": ["true_class", "true_label", "actual_class", "target_class"],
    "predicted_class": [
        "predicted_class", "pred_class", "prediction", "predicted_label"
    ],
    "confidence": [
        "predicted_confidence", "confidence", "max_probability", "probability"
    ],
    "sample_id": [
        "sample_id", "image_id", "filename", "file_name", "path", "index"
    ],
}

def resolve_column(df, candidates, required=True):
    for name in candidates:
        if name in df.columns:
            return name
    if required:
        raise KeyError(
            f"Nenhuma das colunas esperadas foi encontrada: {candidates}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

true_col = resolve_column(holdout_df, COLUMN_CANDIDATES["true_class"])
pred_col = resolve_column(holdout_df, COLUMN_CANDIDATES["predicted_class"])
conf_col = resolve_column(holdout_df, COLUMN_CANDIDATES["confidence"])
id_col = resolve_column(
    holdout_df,
    COLUMN_CANDIDATES["sample_id"],
    required=False
)

working_df = holdout_df.copy()
working_df["_true_canonical"] = working_df[true_col].map(canonicalize_class)
working_df["_pred_canonical"] = working_df[pred_col].map(canonicalize_class)
working_df["_confidence"] = pd.to_numeric(
    working_df[conf_col],
    errors="coerce"
)

working_df = working_df[
    working_df["_true_canonical"] == working_df["_pred_canonical"]
].dropna(subset=["_confidence"])

selected_rows = []

for class_name, group in working_df.groupby("_true_canonical", sort=True):
    median_confidence = group["_confidence"].median()
    chosen = (
        group.assign(
            _distance_to_median=(
                group["_confidence"] - median_confidence
            ).abs()
        )
        .sort_values(
            ["_distance_to_median", "_confidence"],
            ascending=[True, True]
        )
        .iloc[0]
    )
    selected_rows.append(chosen)

selected_cases_df = pd.DataFrame(selected_rows).reset_index(drop=True)

test_cases = []
for row_number, row in selected_cases_df.iterrows():
    raw_id = (
        str(row[id_col])
        if id_col is not None
        else f"holdout_row_{int(row.name)}"
    )

    test_cases.append({
        "sample_id": raw_id,
        "true_class": row["_true_canonical"],
        "predicted_class": row["_pred_canonical"],
        "confidence": float(row["_confidence"]),

        # A CNN classifica a imagem, mas não gera descrição textual da termografia.
        # Mantemos vazio para não introduzir evidência manual.
        "observed_evidence": "",

        # Rastreabilidade
        "cnn_source_row": int(row.name),
        "selection_rule": "correct_prediction_closest_to_class_median_confidence",
    })

expected_classes = set(working_df["_true_canonical"].unique())
selected_classes = {case["predicted_class"] for case in test_cases}

if selected_classes != expected_classes:
    raise RuntimeError(
        "A seleção não representou todas as classes corretamente classificadas. "
        f"Esperadas: {sorted(expected_classes)}; "
        f"selecionadas: {sorted(selected_classes)}"
    )

print(
    f"Hold-out: {len(holdout_df)} amostras | "
    f"corretas: {len(working_df)} | "
    f"casos selecionados: {len(test_cases)}"
)
display(pd.DataFrame(test_cases))


Hold-out: 500 amostras | corretas: 495 | casos selecionados: 6


,sample_id,true_class,predicted_class,confidence,observed_evidence,cnn_source_row,selection_rule
0,holdout_row_0,Bearing_Failure,Bearing_Failure,0.738761,,0,correct_prediction_closest_to_class_median_con...
1,holdout_row_1,Blocked_Rotor,Blocked_Rotor,0.580285,,1,correct_prediction_closest_to_class_median_con...
2,holdout_row_2,Lack_of_Phase,Lack_of_Phase,0.755191,,2,correct_prediction_closest_to_class_median_con...
3,holdout_row_3,Normal Operation,Normal Operation,0.802963,,3,correct_prediction_closest_to_class_median_con...
4,holdout_row_4,Overheating,Overheating,0.792394,,4,correct_prediction_closest_to_class_median_con...
5,holdout_row_5,Ventilation_Defect,Ventilation_Defect,0.893691,,5,correct_prediction_closest_to_class_median_con...


## 10. Adaptador único de carregamento do SLM

Esta célula preserva as diferenças críticas de cada modelo.

In [14]:
# ============================================================
# ADAPTADOR ÚNICO — CARREGAMENTO DIRETO DO SLM
# Não utiliza snapshot_download nem pasta local manual.
# ============================================================

def dtype_from_string(dtype_name: str):
    """Converte o dtype informado na configuração para torch.dtype."""

    if dtype_name == "bfloat16":
        return torch.bfloat16

    if dtype_name == "float32":
        return torch.float32

    return torch.float16


def clear_cuda():
    """Libera memória antes do carregamento do modelo."""

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_slm(
    config: Dict[str, Any],
    hf_token: str
):
    clear_cuda()

    if not hf_token:
        raise RuntimeError(
            "HF_TOKEN não está disponível para carregar o modelo."
        )

    model_name = config["model_name"]

    torch_dtype = dtype_from_string(
        config.get("torch_dtype", "float16")
    )

    print("=" * 70)
    print("Carregando modelo:", model_name)
    print("Modo:", config.get("loading_mode"))
    print("=" * 70)

    # ========================================================
    # NÃO USAMOS MAIS SNAPSHOT LOCAL DO MODELO
    # ========================================================

    # model_slug = model_name.split("/")[-1]
    # local_model_dir = f"/content/models/{model_slug}"

    # model_snapshot_path = snapshot_download(
    #     repo_id=model_name,
    #     local_dir=local_model_dir,
    #     token=hf_token,
    # )

    # O snapshot completo poderia baixar arquivos desnecessários
    # ou depender de várias transferências paralelas.
    # Por isso o modelo volta a ser carregado diretamente pelo
    # identificador oficial do Hugging Face.

    # ========================================================
    # TOKENIZER
    # ========================================================

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=config.get(
            "use_fast_tokenizer",
            True
        ),
        trust_remote_code=config.get(
            "trust_remote_code",
            False
        ),

        # Token passado explicitamente.
        token=hf_token,

        # Não usar local_files_only=True porque o modelo
        # pode ainda não estar integralmente no cache.
        local_files_only=False,
    )

    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            raise RuntimeError(
                "O tokenizer não possui pad_token nem eos_token."
            )

        tokenizer.pad_token = tokenizer.eos_token

    if config.get("padding_side") is not None:
        tokenizer.padding_side = config["padding_side"]

    # Configuração específica do Qwen.
    # Não interfere no Zephyr, Mistral ou TinyLlama.
    if SELECTED_MODEL == "qwen":
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # ========================================================
    # ARGUMENTOS GERAIS DO MODELO
    # ========================================================

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch_dtype,
        "trust_remote_code": config.get(
            "trust_remote_code",
            False
        ),

        # Token explicitamente propagado ao download dos pesos.
        "token": hf_token,

        "low_cpu_mem_usage": True,

        # Não usar local_files_only=True nesta versão.
        "local_files_only": False,
    }

    # ========================================================
    # QUANTIZAÇÃO OPCIONAL EM 4 BITS
    # Usada por Zephyr e Mistral.
    # ========================================================

    if config.get("load_in_4bit", False):

        compute_dtype = dtype_from_string(
            config.get(
                "bnb_4bit_compute_dtype",
                "float16"
            )
        )

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,

            bnb_4bit_quant_type=config.get(
                "bnb_4bit_quant_type",
                "nf4"
            ),

            bnb_4bit_compute_dtype=compute_dtype,

            bnb_4bit_use_double_quant=config.get(
                "bnb_4bit_use_double_quant",
                False
            ),
        )

        model_kwargs["quantization_config"] = bnb_config

        print("Quantização: 4-bit")
        print(
            "Tipo:",
            config.get(
                "bnb_4bit_quant_type",
                "nf4"
            )
        )
        print(
            "Double quant:",
            config.get(
                "bnb_4bit_use_double_quant",
                False
            )
        )

    else:
        print("Quantização: não utilizada")

    # ========================================================
    # CARREGAMENTO DIRETO DO MODELO
    # ========================================================

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )

    model.eval()

    # ========================================================
    # CONFIGURAÇÕES DE GERAÇÃO
    # ========================================================

    model.config.pad_token_id = tokenizer.pad_token_id

    model.generation_config.pad_token_id = (
        tokenizer.pad_token_id
    )

    model.generation_config.eos_token_id = (
        tokenizer.eos_token_id
    )

    model.config.use_cache = True

    # ========================================================
    # VALIDAÇÃO
    # ========================================================

    print("\nModelo carregado com sucesso:", model_name)
    print("Label:", config.get("model_label"))
    print("Modo:", config.get("loading_mode"))
    print("Observação:", config.get("notes"))
    print("Tokenizer:", tokenizer.name_or_path)
    print("Model:", model.config._name_or_path)
    print(
        "EOS:",
        tokenizer.eos_token,
        tokenizer.eos_token_id
    )
    print(
        "PAD:",
        tokenizer.pad_token,
        tokenizer.pad_token_id
    )
    print(
        "Chat template existe?",
        tokenizer.chat_template is not None
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

        print(
            "VRAM alocada:",
            round(
                torch.cuda.memory_allocated() / 1024**3,
                3
            ),
            "GB"
        )

        print(
            "VRAM reservada:",
            round(
                torch.cuda.memory_reserved() / 1024**3,
                3
            ),
            "GB"
        )

    return tokenizer, model


tokenizer, model = load_slm(
    EXPERIMENT_CONFIG,
    hf_token=hf_token
)

Carregando modelo: mistralai/Mistral-7B-Instruct-v0.2
Modo: 4bit_nf4


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


Modelo carregado com sucesso: mistralai/Mistral-7B-Instruct-v0.2
Label: Mistral-7B-Instruct-v0.2
Modo: 4bit_nf4
Observação: Mistral validado em 4-bit NF4; double_quant=False.
Tokenizer: mistralai/Mistral-7B-Instruct-v0.2
Model: mistralai/Mistral-7B-Instruct-v0.2
EOS: </s> 2
PAD: </s> 2
Chat template existe? True
VRAM alocada: 4.588 GB
VRAM reservada: 4.623 GB


In [ ]:
# def dtype_from_string(dtype_name: str):
#     if dtype_name == "bfloat16":
#         return torch.bfloat16
#     if dtype_name == "float32":
#         return torch.float32
#     return torch.float16


# def clear_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()


# def load_slm(config: Dict[str, Any], hf_token: str):
#     clear_cuda()

#     if not hf_token:
#         raise RuntimeError("HF_TOKEN não está disponível para carregar o modelo.")

#     model_name = config["model_name"]
#     torch_dtype = dtype_from_string(
#         config.get("torch_dtype", "float16")
#     )

#     # --------------------------------------------------------
#     # DOWNLOAD COMPLETO DO REPOSITÓRIO PARA UMA PASTA LOCAL
#     # --------------------------------------------------------
#     model_slug = model_name.split("/")[-1]
#     local_model_dir = f"/content/models/{model_slug}"

#     print("Preparando snapshot local do modelo:", model_name)

#     # model_snapshot_path = snapshot_download(
#     #     repo_id=model_name,
#     #     local_dir=local_model_dir,
#     #     local_dir_use_symlinks=False,
#     #     token=hf_token,
#     # )

#     # print("Snapshot do modelo disponível em:")
#     # print(model_snapshot_path)

#     # --------------------------------------------------------
#     # TOKENIZER LOCAL
#     # --------------------------------------------------------
#     tokenizer = AutoTokenizer.from_pretrained(
#         model_snapshot_path,
#         use_fast=config.get("use_fast_tokenizer", True),
#         trust_remote_code=config.get("trust_remote_code", False),
#         local_files_only=True,
#     )

#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     if config.get("padding_side") is not None:
#         tokenizer.padding_side = config["padding_side"]

#     if SELECTED_MODEL == "qwen":
#         tokenizer.pad_token = tokenizer.eos_token
#         tokenizer.pad_token_id = tokenizer.eos_token_id

#     model_kwargs = {
#         "device_map": "auto",
#         "torch_dtype": torch_dtype,
#         "trust_remote_code": config.get(
#             "trust_remote_code",
#             False
#         ),
#         "local_files_only": True,
#         "low_cpu_mem_usage": True,
#     }

#     if config.get("load_in_4bit", False):
#         compute_dtype = dtype_from_string(
#             config.get(
#                 "bnb_4bit_compute_dtype",
#                 "float16"
#             )
#         )

#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type=config.get(
#                 "bnb_4bit_quant_type",
#                 "nf4"
#             ),
#             bnb_4bit_compute_dtype=compute_dtype,
#             bnb_4bit_use_double_quant=config.get(
#                 "bnb_4bit_use_double_quant",
#                 False
#             ),
#         )

#         model_kwargs["quantization_config"] = bnb_config

#     # --------------------------------------------------------
#     # MODELO LOCAL
#     # --------------------------------------------------------
#     model = AutoModelForCausalLM.from_pretrained(
#         model_snapshot_path,
#         **model_kwargs
#     )

#     model.eval()

#     model.config.pad_token_id = tokenizer.pad_token_id
#     model.generation_config.pad_token_id = tokenizer.pad_token_id
#     model.generation_config.eos_token_id = tokenizer.eos_token_id

#     print("Modelo carregado localmente com sucesso:", model_name)
#     print("Label:", config.get("model_label"))
#     print("Modo:", config.get("loading_mode"))
#     print("Observação:", config.get("notes"))
#     print("Tokenizer:", tokenizer.name_or_path)
#     print("Model:", model.config._name_or_path)
#     print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)
#     print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)
#     print("Chat template existe?", tokenizer.chat_template is not None)

#     if torch.cuda.is_available():
#         print(
#             "VRAM alocada:",
#             round(torch.cuda.memory_allocated() / 1024**3, 3),
#             "GB"
#         )
#         print(
#             "VRAM reservada:",
#             round(torch.cuda.memory_reserved() / 1024**3, 3),
#             "GB"
#         )

#     return tokenizer, model


# tokenizer, model = load_slm(
#     EXPERIMENT_CONFIG,
#     hf_token=hf_token
# )


## 11. Geração padronizada com regras específicas por modelo

In [15]:
def is_invalid_report(
    report: str,
    profile: str,
    minimum_chars: int = 120
) -> bool:

    text = report.strip()
    low = text.lower()

    forbidden_placeholders = [
        "[insert",
        "insert date",
        "insert location",
        "insert equipment",
        "insert temperature",
        "[date]",
        "[location]",
        "[equipment",
    ]

    generic_invalid = (
        len(text) < minimum_chars
        or text in ["", "</s>", "<s>"]
        or "traceback" in low
        or "cuda" in low
        or any(item in low for item in forbidden_placeholders)
    )

    if generic_invalid:
        return True

    if profile == "mistral":
        return "<unk>" in text or "ACHE" in text

    if profile == "qwen":
        return text.count("!") > 20

    if profile == "tinyllama":
        return (
            text.count("!") > 10
            or text.count("#") > 10
            or len(set(text.replace(" ", ""))) < 10
            or "- - -" in text
            or text.count("\n      -") > 20
            or text.count("\n        -") > 20
            or text.count("Fault Type:") > 3
            or text.count("Risk Level:") > 3
            or "technical maintenance report for" in low
            or "\ndate:" in low
            or "\nlocation:" in low
        )

    return False
# def is_invalid_report(report: str, profile: str, minimum_chars: int = 120) -> bool:
#     text = report.strip()
#     low = text.lower()

#     generic_invalid = (
#         len(text) < minimum_chars
#         or text in ["", "</s>", "<s>"]
#         or "Traceback" in text
#         or "CUDA" in text
#     )

#     if generic_invalid:
#         return True

#     if profile == "mistral":
#         return "<unk>" in text or "ACHE" in text

#     if profile == "qwen":
#         return text.count("!") > 20

#     if profile == "tinyllama":
#         return (
#             text.count("!") > 10
#             or text.count("#") > 10
#             or len(set(text.replace(" ", ""))) < 10
#             or "- - -" in text
#             or text.count("\n      -") > 20
#             or text.count("\n        -") > 20
#             or text.count("Fault Type:") > 3
#             or text.count("Risk Level:") > 3
#         )

#     return False


def apply_model_chat_template(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt.strip()}]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    return prompt.strip()


def configure_seed(seed: int) -> None:
    set_seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def generate_report(
    prompt: str,
    config: Dict[str, Any],
    seed: int
) -> Dict[str, Any]:

    configure_seed(seed)
    input_text = apply_model_chat_template(prompt)

    raw_inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=False,
        truncation=False,
    )

    raw_input_tokens = int(raw_inputs["input_ids"].shape[-1])
    max_input_tokens = int(config["max_input_tokens"])
    input_was_truncated = raw_input_tokens > max_input_tokens

    if input_was_truncated:
        raise RuntimeError(
            f"Prompt excedeu o limite final: "
            f"{raw_input_tokens} > {max_input_tokens} tokens. "
            "Nenhuma geração foi executada."
        )

    inputs = {
        key: value.to(model.device)
        for key, value in raw_inputs.items()
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    generation_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs.get("attention_mask"),
        "max_new_tokens": int(config["max_new_tokens"]),
        "do_sample": bool(config["do_sample"]),
        "repetition_penalty": float(config["repetition_penalty"]),
        "pad_token_id": (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
        "return_dict_in_generate": True,
    }

    if generation_kwargs["do_sample"]:
        if config.get("temperature") is not None:
            generation_kwargs["temperature"] = float(config["temperature"])
        if config.get("top_p") is not None:
            generation_kwargs["top_p"] = float(config["top_p"])
        if config.get("top_k") is not None:
            generation_kwargs["top_k"] = int(config["top_k"])

    generation_kwargs = {
        key: value
        for key, value in generation_kwargs.items()
        if value is not None
    }

    start = time.perf_counter()
    with torch.inference_mode():
        generation_output = model.generate(**generation_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    sequences = generation_output.sequences
    generated_ids = sequences[0][raw_input_tokens:]

    output_tokens = int(generated_ids.shape[-1])
    eos_id = tokenizer.eos_token_id
    ended_with_eos = bool(
        eos_id is not None
        and (generated_ids == eos_id).any().item()
    )

    reached_token_limit = (
        output_tokens >= int(config["max_new_tokens"])
    )
    was_truncated = bool(
        reached_token_limit and not ended_with_eos
    )

    if ended_with_eos:
        finish_reason = "eos_token"
    elif reached_token_limit:
        finish_reason = "length"
    else:
        finish_reason = "other"

    report = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    peak_vram_gb = None
    if torch.cuda.is_available():
        peak_vram_gb = round(
            torch.cuda.max_memory_allocated() / 1024**3,
            3
        )

    valid_report = (
        not is_invalid_report(
            report,
            config.get("invalid_rules", "generic"),
            minimum_chars=int(
                config.get("minimum_report_chars", 120)
            ),
        )
        and not was_truncated
        and not input_was_truncated
    )

    return {
        "report": report,
        "valid_report": valid_report,
        "latency_seconds": latency,
        "input_tokens": raw_input_tokens,
        "raw_input_tokens": raw_input_tokens,
        "output_tokens": output_tokens,
        "peak_vram_gb": peak_vram_gb,
        "seed": seed,
        "finish_reason": finish_reason,
        "ended_with_eos": ended_with_eos,
        "reached_token_limit": reached_token_limit,
        "was_truncated": was_truncated,
        "input_was_truncated": input_was_truncated,
    }


## 12. Teste mínimo obrigatório antes dos 6 casos

In [16]:

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    test_prompt = """
Predicted fault: Bearing Failure
CNN confidence: 98.7%

Retrieved evidence:
- Elevated temperature near the bearing housing.
- Possible lubrication degradation.

Generate a concise technical maintenance report with diagnosis, evidence, and recommended actions.
"""

    test_gen = generate_report(
        test_prompt,
        EXPERIMENT_CONFIG,
        seed=EXPERIMENT_CONFIG["base_seed"] - 1,
    )

    print("valid_report =", test_gen["valid_report"])
    print("finish_reason =", test_gen["finish_reason"])
    print("was_truncated =", test_gen["was_truncated"])
    print("latency_seconds =", round(test_gen["latency_seconds"], 2))
    print("input_tokens =", test_gen["input_tokens"])
    print("output_tokens =", test_gen["output_tokens"])
    print("peak_vram_gb =", test_gen.get("peak_vram_gb"))
    print("-" * 80)
    print(test_gen["report"])


## 13. Execução dos seis casos

In [17]:

def safe_get_fault_type(doc):
    return doc.get("fault_type", doc.get("type", doc.get("id", "NA")))

def safe_get_section(doc):
    return doc.get("section", "full_entry")

def safe_get_score(doc):
    return doc.get("score", 0.0)

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def run_experiment(
    cases: List[Dict[str, Any]],
    config: Dict[str, Any]
) -> pd.DataFrame:

    results = []

    for case_index, case in enumerate(
        tqdm(cases, desc=config.get("model_label", config.get("model_name")))
    ):
        case_seed = int(config["base_seed"]) + case_index

        canonical_class = canonicalize_class(case["predicted_class"])

        retrieved = retrieve_context(
            predicted_class=canonical_class,
            confidence=case["confidence"],
            observed_evidence=case.get("observed_evidence", ""),
            top_k=config["rag_top_k"]
        )

        prompt, retrieved_context_text, context_budget_metadata = build_prompt(
            predicted_class=canonical_class,
            confidence=case["confidence"],
            retrieved_docs=retrieved,
            observed_evidence=case.get("observed_evidence", "")
        )
        retrieved_context_json = json.dumps(
            retrieved,
            ensure_ascii=False
        )

        gen = generate_report(
            prompt,
            config,
            seed=case_seed,
        )

        row = {
            **case,
            "predicted_class_original": case["predicted_class"],
            "predicted_class": canonical_class,
            "model_name": config["model_name"],
            "model_label": config.get("model_label"),
            "loading_mode": config.get("loading_mode"),
            "experiment_version": config.get("experiment_version"),
            "temperature": config.get("temperature"),
            "top_p": config.get("top_p"),
            "top_k": config.get("top_k"),
            "do_sample": config.get("do_sample"),
            "repetition_penalty": config.get("repetition_penalty"),
            "max_new_tokens": config.get("max_new_tokens"),
            "max_input_tokens": config.get("max_input_tokens"),
            "rag_top_k": config["rag_top_k"],
            "retrieved_faults": " | ".join(
                safe_get_fault_type(d) for d in retrieved
            ),
            "retrieved_sections": " | ".join(
                safe_get_section(d) for d in retrieved
            ),
            "retrieval_scores": " | ".join(
                f"{safe_get_score(d):.4f}" for d in retrieved
            ),
            "retrieved_context": retrieved_context_text,
            "retrieved_context_json": retrieved_context_json,
            "context_budget_json": json.dumps(
                context_budget_metadata,
                ensure_ascii=False
            ),
            "retrieved_context_sha256": stable_hash(retrieved_context_text),
            "prompt": prompt,
            "prompt_sha256": stable_hash(prompt),
            "context_was_trimmed": any(
                item["document_was_trimmed"]
                for item in context_budget_metadata["documents"]
            ),
            **gen,
        }

        results.append(row)

    return pd.DataFrame(results)


results_df = run_experiment(test_cases, EXPERIMENT_CONFIG)

# Exportação em CSV
results_df.to_csv(
    EXPERIMENT_CONFIG["output_csv"],
    index=False,
    encoding="utf-8"
)

# Exportação em JSON, preservando textos e estruturas longas
with open(EXPERIMENT_CONFIG["output_json"], "w", encoding="utf-8") as f:
    json.dump(
        results_df.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2,
    )

# Manifesto de reprodutibilidade
manifest = {
    "experiment_version": EXPERIMENT_CONFIG["experiment_version"],
    "selected_model": SELECTED_MODEL,
    "model_name": EXPERIMENT_CONFIG["model_name"],
    "model_label": EXPERIMENT_CONFIG["model_label"],
    "loading_mode": EXPERIMENT_CONFIG["loading_mode"],
    "generation_config": {
        key: EXPERIMENT_CONFIG.get(key)
        for key in [
            "max_new_tokens",
            "do_sample",
            "temperature",
            "top_p",
            "top_k",
            "repetition_penalty",
            "base_seed",
            "max_input_tokens",
            "prompt_safety_margin",
        ]
    },
    "rag_config": {
        "embedding_model": EXPERIMENT_CONFIG["embedding_model"],
        "rag_top_k": EXPERIMENT_CONFIG["rag_top_k"],
        "rag_json_path": EXPERIMENT_CONFIG["rag_json_path"],
        "number_of_documents": len(documents),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "files": {
        "csv": EXPERIMENT_CONFIG["output_csv"],
        "json": EXPERIMENT_CONFIG["output_json"],
    },
}

with open(EXPERIMENT_CONFIG["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("CSV salvo em:", EXPERIMENT_CONFIG["output_csv"])
print("JSON salvo em:", EXPERIMENT_CONFIG["output_json"])
print("Manifesto salvo em:", EXPERIMENT_CONFIG["manifest_json"])

display_columns = [
    "sample_id",
    "predicted_class",
    "model_label",
    "latency_seconds",
    "input_tokens",
    "output_tokens",
    "peak_vram_gb",
    "finish_reason",
    "was_truncated",
    "input_was_truncated",
    "valid_report",
    "retrieved_faults",
]
display(results_df[display_columns])

# Validação obrigatória da rodada
expected_cases = len(test_cases)
problems = []

if len(results_df) != expected_cases:
    problems.append(
        f"Número incorreto de casos: {len(results_df)}; esperado: {expected_cases}."
    )

if results_df["sample_id"].nunique() != expected_cases:
    problems.append("Há sample_id ausente ou duplicado.")

if results_df["report"].fillna("").str.strip().eq("").any():
    problems.append("Há relatório vazio.")

if results_df["was_truncated"].astype(bool).any():
    bad = results_df.loc[
        results_df["was_truncated"].astype(bool),
        "sample_id"
    ].tolist()
    problems.append(f"Relatórios truncados: {bad}")

if results_df["input_was_truncated"].astype(bool).any():
    bad = results_df.loc[
        results_df["input_was_truncated"].astype(bool),
        "sample_id"
    ].tolist()
    problems.append(f"Prompts truncados: {bad}")

if not results_df["valid_report"].astype(bool).all():
    bad = results_df.loc[
        ~results_df["valid_report"].astype(bool),
        "sample_id"
    ].tolist()
    problems.append(f"Relatórios inválidos: {bad}")

print("\n" + "=" * 72)
if problems:
    print("RODADA NÃO VALIDADA")
    for item in problems:
        print(" -", item)

    if EXPERIMENT_CONFIG.get("fail_on_truncation", True):
        raise RuntimeError(
            "A execução terminou, mas falhou na validação. "
            "Não use este CSV no notebook de métricas."
        )
else:
    print("RODADA VALIDADA")
    print(f"{expected_cases}/{expected_cases} relatórios completos e válidos.")
    print("Nenhum prompt ou relatório foi truncado.")
    print("Arquivos prontos para o notebook de avaliação.")
print("=" * 72)


Mistral-7B-Instruct-v0.2:   0%|          | 0/6 [00:00<?, ?it/s]

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


CSV salvo em: /content/results_slm_rag_final/slm_rag_results_mistral_final.csv
JSON salvo em: /content/results_slm_rag_final/slm_rag_results_mistral_final.json
Manifesto salvo em: /content/results_slm_rag_final/manifest_mistral_final.json


,sample_id,predicted_class,model_label,latency_seconds,input_tokens,output_tokens,peak_vram_gb,finish_reason,was_truncated,input_was_truncated,valid_report,retrieved_faults
0,holdout_row_0,Bearing_Failure,Mistral-7B-Instruct-v0.2,16.053341,1237,201,4.956,eos_token,False,False,True,Bearing Failure | Normal Operation | Phase Loss
1,holdout_row_1,Blocked_Rotor,Mistral-7B-Instruct-v0.2,23.477209,1235,309,4.957,eos_token,False,False,True,Blocked Rotor | Phase Loss | Normal Operation
2,holdout_row_2,Lack_of_Phase,Mistral-7B-Instruct-v0.2,18.656068,1236,249,4.957,eos_token,False,False,True,Phase Loss | Normal Operation | Overheating
3,holdout_row_3,Normal Operation,Mistral-7B-Instruct-v0.2,18.875420,1235,219,4.957,eos_token,False,False,True,Normal Operation | Phase Loss | Overheating
4,holdout_row_4,Overheating,Mistral-7B-Instruct-v0.2,13.620956,1236,177,4.957,eos_token,False,False,True,Overheating | Ventilation Defect | Normal Oper...
5,holdout_row_5,Ventilation_Defect,Mistral-7B-Instruct-v0.2,15.019636,1235,196,4.957,eos_token,False,False,True,Ventilation Defect | Overheating | Normal Oper...



RODADA VALIDADA
6/6 relatórios completos e válidos.
Nenhum prompt ou relatório foi truncado.
Arquivos prontos para o notebook de avaliação.


## 14. Visualizar relatório gerado

In [18]:
i = 0
print("Classe prevista:", results_df.loc[i, "predicted_class"])
print("Confiança:", results_df.loc[i, "confidence"])
print("Modelo:", results_df.loc[i, "model_label"])
print("Latência:", results_df.loc[i, "latency_seconds"])
print("VRAM pico:", results_df.loc[i, "peak_vram_gb"])
print("-" * 80)
print(results_df.loc[i, "report"])

Classe prevista: Bearing_Failure
Confiança: 0.7387608
Modelo: Mistral-7B-Instruct-v0.2
Latência: 16.053341277999834
VRAM pico: 4.956
--------------------------------------------------------------------------------
1. Technical diagnosis:
Based on the CNN prediction, a potential bearing failure is indicated in the induction motor.

2. Supporting evidence:
The CNN model identified a bearing failure with a confidence of 0.739.

3. Probable causes:
Possible root causes include insufficient lubricant, excessive lubrication, ingress of contaminants, misalignment, or excessive load.

4. Recommended maintenance actions:
Inspect the bearing for signs of damage, such as increased vibration or localized heating. Check lubrication levels and quality. Align motor and driven load properly. Monitor load conditions to prevent overloading.

5. Severity and risk assessment:
A bearing failure can result in significant motor damage, increased vibration, acoustic noise, and potential secondary electrical o

## 15. Grid exploratório antigo

Desativado na rodada final. Os parâmetros de geração já estão congelados e padronizados.


In [ ]:

# ============================================================
# GRID EXPLORATÓRIO ANTIGO — NÃO EXECUTAR NA RODADA FINAL
# ============================================================
# A rodada final já possui parâmetros congelados e iguais entre os modelos.
# Esta célula é mantida apenas para documentação histórica.

RUN_GRID = False

if RUN_GRID:
    raise RuntimeError(
        "O grid exploratório foi desativado para a rodada final controlada."
    )


In [19]:
# ============================================================
# GRID SEARCH FINAL — QWEN / CNN REAL / 6 CASOS
# ============================================================

from pathlib import Path
import pandas as pd
import json

# Mantemos as três configurações avaliadas anteriormente.
# O valor final de 700 tokens é preservado em todas as combinações
# para evitar que o tamanho máximo da resposta seja uma variável.
param_grid = [
    {
        "temperature": 0.1,
        "top_p": 0.90,
        "max_new_tokens": 700,
        "do_sample": True,
    },
    {
        "temperature": 0.2,
        "top_p": 0.90,
        "max_new_tokens": 700,
        "do_sample": True,
    },
    {
        "temperature": 0.5,
        "top_p": 0.95,
        "max_new_tokens": 700,
        "do_sample": True,
    },
]

# Usa os mesmos seis casos reais selecionados do hold-out da CNN.
selected_cases = test_cases

grid_results = []

print("=" * 80)
print("INICIANDO GRID SEARCH")
print(f"Modelo: {EXPERIMENT_CONFIG['model_label']}")
print(f"Casos por configuração: {len(selected_cases)}")
print(f"Configurações: {len(param_grid)}")
print(f"Total de gerações: {len(selected_cases) * len(param_grid)}")
print("=" * 80)

for grid_id, params in enumerate(param_grid, start=1):

    print("\n" + "=" * 80)
    print(f"CONFIGURAÇÃO {grid_id}/{len(param_grid)}")
    print(json.dumps(params, indent=2, ensure_ascii=False))
    print("=" * 80)

    run_config = EXPERIMENT_CONFIG.copy()
    run_config.update(params)

    # Seed-base diferente por configuração, mas determinístico.
    run_config["base_seed"] = (
        int(EXPERIMENT_CONFIG.get("base_seed", 42))
        + (grid_id - 1) * 100
    )

    df_tmp = run_experiment(
        selected_cases,
        run_config,
    )

    # Identificação explícita da combinação.
    df_tmp["grid_id"] = grid_id
    df_tmp["temperature"] = params["temperature"]
    df_tmp["top_p"] = params["top_p"]
    df_tmp["max_new_tokens"] = params["max_new_tokens"]
    df_tmp["do_sample"] = params["do_sample"]
    df_tmp["grid_base_seed"] = run_config["base_seed"]

    grid_results.append(df_tmp)

# Consolidação das 18 gerações.
grid_df = pd.concat(
    grid_results,
    ignore_index=True,
)

# ------------------------------------------------------------
# CAMINHOS DE SAÍDA
# ------------------------------------------------------------
results_dir = Path(
    EXPERIMENT_CONFIG.get(
        "results_dir",
        "/content/results_slm_rag_final",
    )
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

selected_model = globals().get(
    "SELECTED_MODEL",
    "qwen",
)

grid_csv_path = (
    results_dir
    / f"slm_rag_grid_{selected_model}_final.csv"
)

grid_json_path = (
    results_dir
    / f"slm_rag_grid_{selected_model}_final.json"
)

# ------------------------------------------------------------
# SALVAMENTO
# ------------------------------------------------------------
grid_df.to_csv(
    grid_csv_path,
    index=False,
    encoding="utf-8",
)

grid_df.to_json(
    grid_json_path,
    orient="records",
    force_ascii=False,
    indent=2,
)

print("\n" + "=" * 80)
print("GRID SEARCH CONCLUÍDO")
print("=" * 80)
print("CSV salvo em:")
print(grid_csv_path)
print("\nJSON salvo em:")
print(grid_json_path)
print(f"\nTotal de linhas salvas: {len(grid_df)}")

# ------------------------------------------------------------
# VALIDAÇÃO DO GRID
# ------------------------------------------------------------
expected_rows = len(param_grid) * len(selected_cases)

if len(grid_df) != expected_rows:
    raise RuntimeError(
        f"O grid deveria conter {expected_rows} linhas, "
        f"mas foram geradas {len(grid_df)}."
    )

if "input_was_truncated" in grid_df.columns:
    truncated_inputs = int(
        grid_df["input_was_truncated"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
else:
    truncated_inputs = -1

if "was_truncated" in grid_df.columns:
    truncated_outputs = int(
        grid_df["was_truncated"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
else:
    truncated_outputs = -1

if "valid_report" in grid_df.columns:
    valid_reports = int(
        grid_df["valid_report"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
else:
    valid_reports = -1

print("\nVALIDAÇÃO")
print(f"Linhas esperadas:           {expected_rows}")
print(f"Linhas obtidas:             {len(grid_df)}")
print(f"Entradas truncadas:         {truncated_inputs}")
print(f"Saídas truncadas:           {truncated_outputs}")
print(f"Relatórios válidos:         {valid_reports}/{len(grid_df)}")

# ------------------------------------------------------------
# QUADRO RESUMIDO
# ------------------------------------------------------------
display_columns = [
    "grid_id",
    "sample_id",
    "predicted_class",
    "temperature",
    "top_p",
    "max_new_tokens",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "peak_vram_gb",
    "finish_reason",
    "input_was_truncated",
    "was_truncated",
    "valid_report",
]

display_columns = [
    column
    for column in display_columns
    if column in grid_df.columns
]

display(grid_df[display_columns])

print("\nRESUMO POR CONFIGURAÇÃO")

aggregation = {
    "latency_seconds": ["mean", "std"],
    "input_tokens": ["mean", "std"],
    "output_tokens": ["mean", "std"],
}

if "peak_vram_gb" in grid_df.columns:
    aggregation["peak_vram_gb"] = ["mean", "std"]

if "valid_report" in grid_df.columns:
    aggregation["valid_report"] = ["sum", "mean"]

grid_summary = (
    grid_df
    .groupby(
        [
            "grid_id",
            "temperature",
            "top_p",
            "max_new_tokens",
        ],
        dropna=False,
    )
    .agg(aggregation)
    .round(3)
)

display(grid_summary)

INICIANDO GRID SEARCH
Modelo: Mistral-7B-Instruct-v0.2
Casos por configuração: 6
Configurações: 3
Total de gerações: 18

CONFIGURAÇÃO 1/3
{
  "temperature": 0.1,
  "top_p": 0.9,
  "max_new_tokens": 700,
  "do_sample": true
}


Mistral-7B-Instruct-v0.2:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



CONFIGURAÇÃO 2/3
{
  "temperature": 0.2,
  "top_p": 0.9,
  "max_new_tokens": 700,
  "do_sample": true
}


Mistral-7B-Instruct-v0.2:   0%|          | 0/6 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



GRID SEARCH CONCLUÍDO
CSV salvo em:
/content/results_slm_rag_final/slm_rag_grid_mistral_final.csv

JSON salvo em:
/content/results_slm_rag_final/slm_rag_grid_mistral_final.json

Total de linhas salvas: 18

VALIDAÇÃO
Linhas esperadas:           18
Linhas obtidas:             18
Entradas truncadas:         0
Saídas truncadas:           0
Relatórios válidos:         18/18


,grid_id,sample_id,predicted_class,temperature,top_p,max_new_tokens,input_tokens,output_tokens,latency_seconds,peak_vram_gb,finish_reason,input_was_truncated,was_truncated,valid_report
0,1,holdout_row_0,Bearing_Failure,0.1,0.90,700,1237,193,14.774374,4.957,eos_token,False,False,True
1,1,holdout_row_1,Blocked_Rotor,0.1,0.90,700,1235,301,23.531561,4.957,eos_token,False,False,True
2,1,holdout_row_2,Lack_of_Phase,0.1,0.90,700,1236,247,19.202264,4.957,eos_token,False,False,True
3,1,holdout_row_3,Normal Operation,0.1,0.90,700,1235,198,15.280765,4.957,eos_token,False,False,True
4,1,holdout_row_4,Overheating,0.1,0.90,700,1236,154,12.017724,4.957,eos_token,False,False,True
5,1,holdout_row_5,Ventilation_Defect,0.1,0.90,700,1235,198,15.057096,4.957,eos_token,False,False,True
6,2,holdout_row_0,Bearing_Failure,0.2,0.90,700,1237,224,17.016311,4.957,eos_token,False,False,True
7,2,holdout_row_1,Blocked_Rotor,0.2,0.90,700,1235,291,21.522945,4.957,eos_token,False,False,True
8,2,holdout_row_2,Lack_of_Phase,0.2,0.90,700,1236,237,18.382025,4.957,eos_token,False,False,True
9,2,holdout_row_3,Normal Operation,0.2,0.90,700,1235,161,12.835311,4.957,eos_token,False,False,True



RESUMO POR CONFIGURAÇÃO


latency_seconds        input_tokens  \
                                                    mean    std         mean   
grid_id temperature top_p max_new_tokens                                       
1       0.1         0.90  700                     16.644  4.081     1235.667   
2       0.2         0.90  700                     16.603  3.087     1235.667   
3       0.5         0.95  700                     18.306  3.193     1235.667   

                                                output_tokens          \
                                            std          mean     std   
grid_id temperature top_p max_new_tokens                                
1       0.1         0.90  700             0.816       215.167  51.386   
2       0.2         0.90  700             0.816       216.333  45.430   
3       0.5         0.95  700             0.816       241.167  43.245   

                                         peak_vram_gb      valid_report       
                                                 mean  std          sum mean  
grid_id temperature top_p max_new_tokens                                      
1       0.1         0.90  700                   4.957  0.0            6  1.0  
2       0.2         0.90  700                   4.957  0.0            6  1.0  
3       0.5         0.95  700                   4.957  0.0            6  1.0

## 16. Métricas locais opcionais

A avaliação científica final deve ser feita no notebook consolidado de métricas, usando os CSVs finais validados.


In [ ]:
# def add_metrics_if_reference_exists(
#     df: pd.DataFrame,
#     generated_col: str = "report",
#     reference_col: str = "reference_report"
# ) -> pd.DataFrame:
#     df = df.copy()

#     if reference_col not in df.columns:
#         print(f"Coluna '{reference_col}' não encontrada. Métricas textuais não calculadas.")
#         return df

#     from sklearn.metrics.pairwise import cosine_similarity
#     import evaluate

#     metric_embedding_model = SentenceTransformer(EXPERIMENT_CONFIG["embedding_model"])
#     bleu = evaluate.load("bleu")
#     rouge = evaluate.load("rouge")

#     cosine_scores = []
#     bleu_scores = []
#     rouge_l_scores = []

#     for _, row in tqdm(df.iterrows(), total=len(df)):
#         generated = str(row[generated_col])
#         reference = str(row[reference_col])

#         emb = metric_embedding_model.encode(
#             [generated, reference],
#             convert_to_numpy=True,
#             normalize_embeddings=True
#         )
#         cosine_scores.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))

#         bleu_scores.append(float(bleu.compute(predictions=[generated], references=[[reference]])["bleu"]))
#         rouge_l_scores.append(float(rouge.compute(predictions=[generated], references=[reference])["rougeL"]))

#     df["semantic_cosine_similarity"] = cosine_scores
#     df["bleu"] = bleu_scores
#     df["rougeL"] = rouge_l_scores
#     return df


# results_df = add_metrics_if_reference_exists(results_df)
# results_df.to_csv(EXPERIMENT_CONFIG["output_csv"], index=False, encoding="utf-8")
# print("CSV atualizado:", EXPERIMENT_CONFIG["output_csv"])

## 17. Comparação posterior entre modelos

Depois de rodar cada modelo em runtimes separados, faça upload dos CSVs ou mantenha-os em `/content/results_slm_rag` e execute a célula abaixo.

In [ ]:
# from pathlib import Path

# csv_files = sorted(Path(EXPERIMENT_CONFIG["results_dir"]).glob("slm_rag_results_*.csv"))
# print("CSVs encontrados:", [p.name for p in csv_files])

# if csv_files:
#     comparison_df = pd.concat([pd.read_csv(p) for p in csv_files], ignore_index=True)

#     summary_cols = {
#         "latency_seconds": "mean",
#         "input_tokens": "mean",
#         "output_tokens": "mean",
#         "peak_vram_gb": "max",
#         "valid_report": "mean",
#     }

#     for optional in ["semantic_cosine_similarity", "bleu", "rougeL"]:
#         if optional in comparison_df.columns:
#             summary_cols[optional] = "mean"

#     summary_df = comparison_df.groupby("model_label", as_index=False).agg(summary_cols)
#     display(summary_df)
# else:
#     print("Nenhum CSV encontrado ainda.")

In [ ]:
# ============================================================
# RESUMO DA CONFIGURAÇÃO FINAL E DA EXECUÇÃO
# ============================================================

summary_columns = [
    "sample_id",
    "predicted_class",
    "confidence",
    "temperature",
    "top_p",
    "max_new_tokens",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "peak_vram_gb",
    "finish_reason",
    "input_was_truncated",
    "was_truncated",
    "valid_report",
]

execution_summary = results_df[summary_columns].copy()

print("=" * 80)
print("CONFIGURAÇÃO FINAL SELECIONADA")
print("=" * 80)
print(f"Modelo:             {EXPERIMENT_CONFIG['model_label']}")
print(f"Modo de execução:   {EXPERIMENT_CONFIG['loading_mode']}")
print(f"Temperature:        {EXPERIMENT_CONFIG['temperature']}")
print(f"Top-p:              {EXPERIMENT_CONFIG['top_p']}")
print(f"Max new tokens:     {EXPERIMENT_CONFIG['max_new_tokens']}")
print(f"Max input tokens:   {EXPERIMENT_CONFIG['max_input_tokens']}")
print(f"RAG top-k:          {EXPERIMENT_CONFIG['rag_top_k']}")
print("=" * 80)

display(execution_summary)

print("\nRESUMO MÉDIO")
display(
    results_df[
        [
            "latency_seconds",
            "input_tokens",
            "output_tokens",
            "peak_vram_gb",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

## 18. Entrada futura da CNN

Nesta versão, os casos ainda entram como `predicted_class` e `confidence`, preservando a comparação entre SLMs. Depois o classificador com 5-fold poderá preencher automaticamente esses dois campos.

In [ ]:
# Fluxo definitivo:
# holdout_test_predictions.csv
#   -> seleção automática de uma amostra correta por classe
#   -> classe prevista + confiança reais da CNN
#   -> recuperação top-k=3 na base RAG original com 6 documentos
#   -> limitação determinística somente do contexto fornecido
#   -> geração sem truncamento silencioso
#   -> CSV + JSON + manifesto para auditoria e métricas
